In [ ]:
import jax
# MUST be set before any JAX arrays are created to prevent precision loss at 1e-6
jax.config.update("jax_enable_x64", True) 

import jax.numpy as jnp
import optax
from functools import partial
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import configuration and all modular physics/optimization components
from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_operators
from src.control.pulses import make_gaussian_kernel, apply_low_pass_filter
from src.calibration.cz_training import compute_cz_loss
from src.calibration.training_loops import train_pulse

# Fitted parameters for the CZ gate
cal_g, cal_d, cal_a = np.load("../artifacts/cal_params.npy")

In [ ]:
# --- Helper Functions ---
def make_cz_optimizer(learning_rate, steps):
    schedule = optax.cosine_decay_schedule(init_value=learning_rate, decay_steps=steps, alpha=1e-3)
    return optax.chain(optax.clip_by_global_norm(1.0), optax.adam(schedule))

def make_setup(config, n_harmonics=8, key=None):
    key = jax.random.PRNGKey(0) if key is None else key
    dt, T = config.pulse.dt_ns, config.pulse.duration_ns
    n_steps = int(round(T / dt))

    H0, Hz = build_two_qubit_operators(config.hardware.levels, cal_a, cal_a, cal_g,idle_detuning_mhz=600.0)
    x = (jnp.arange(n_steps) + 0.5) / n_steps
    basis = jnp.sin(jnp.pi * x[:, None] * jnp.arange(1, n_harmonics + 1)[None, :])
    kernel = make_gaussian_kernel(dt, config.pulse.cutoff_freq_ghz)
    
    f_max = 2 * jnp.pi * 0.6

    # start near the |11>-|20> crossing: lower qubit 1 by ~300 MHz = 1.9 rad/ns, as a half-sine
    init = {'flux_coeffs': jnp.zeros(n_harmonics).at[0].set(-2.0) + 0.05 * jax.random.normal(key, (n_harmonics,)),
            'phases': jnp.zeros(2)}
    loss_fn = partial(compute_cz_loss, dt=dt, H0=H0, Hz_control=Hz, basis=basis, kernel=kernel, f_max=f_max,
                      penalty_weight=config.optimizer.leakage_weight)
    return H0, Hz, loss_fn, init, basis, kernel, f_max

In [ ]:
# --- Main Setup and Execution ---
config = ExperimentConfig.load("configs/two_qubit_CZ.yaml")

N_HARMONICS = 8

H0, Hz, bound_loss_fn, init_params, basis, kernel, f_max = make_setup(config, n_harmonics=N_HARMONICS)
optimizer = make_cz_optimizer(0.01, config.optimizer.steps)


print("Optimizing Two-Qubit CZ Gate...")
opt_params, history = train_pulse(bound_loss_fn, init_params, optimizer, config.optimizer.steps)

In [ ]:
# --- Plotting ---
tlist = np.arange(0, config.pulse.duration_ns, config.pulse.dt_ns)

# Reconstruct the flux pulse and convert back to GHz
flux_rad = f_max * jnp.tanh(basis @ opt_params['flux_coeffs'] / f_max)
final_flux_ghz = apply_low_pass_filter(flux_rad, kernel) / (2 * jnp.pi)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'], label='Total Loss', color='black')
ax1.plot(history['infidelity'], label='Infidelity', linestyle='--')
ax1.plot(history['leakage'], label='Leakage', linestyle='-.')
ax1.set_yscale('log')
ax1.set_xlabel('Optimization Step')
ax1.set_ylabel('Metric Value')
ax1.legend()
ax1.set_title('CZ Gate Training History')
ax1.grid(True)

ax2.plot(tlist, final_flux_ghz, color='purple', label='Filtered Flux Pulse')
ax2.set_xlabel('Time (ns)')
ax2.set_ylabel('Flux Excursion (GHz)')
ax2.set_title('Bandwidth-Limited Flux Control')
#ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Extracting the Unitary Evolution History ---
@jax.jit
def get_unitary_history(params, dt, H0, Hz_control, basis, kernel, f_max):
    """Replays the optimized pulse and saves the matrix at every time step."""
    raw = basis @ params['flux_coeffs']
    flux = f_max * jnp.tanh(raw / f_max)
    flux = apply_low_pass_filter(flux, kernel)

    def step(U, flux_val):
        H = H0 + flux_val * Hz_control
        U_next = jax.scipy.linalg.expm(-1j * H * dt) @ U
        return U_next, U_next 
    
    # 9x9 Identity matrix as the starting state
    U_init = jnp.eye(H0.shape[0], dtype=jnp.complex128)
    _, U_history = jax.lax.scan(step, U_init, flux)
    return U_history

U_hist = get_unitary_history(opt_params, config.pulse.dt_ns, H0, Hz, basis, kernel, f_max)

# --- Calculating Populations and Phase ---
# In our 3x3 Kronecker product: |00>=0, |01>=1, |02>=2, |10>=3, |11>=4, |12>=5, |20>=6
# When starting in |11> (index 4), track population of |11> and the leakage state |20> (index 6)
pop_11 = jnp.abs(U_hist[:, 4, 4])**2
pop_20 = jnp.abs(U_hist[:, 6, 4])**2

# Isolate the conditional phase (in radians, unwrapped to prevent jumping at pi/-pi)
phase_cond_rad = jnp.unwrap(
    jnp.angle(U_hist[:, 4, 4]) - jnp.angle(U_hist[:, 1, 1]) - 
    jnp.angle(U_hist[:, 3, 3]) + jnp.angle(U_hist[:, 0, 0])
)
# Convert to units of Pi
phase_cond_pi = phase_cond_rad / jnp.pi

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# The Avoided Crossing Collision
ax1.plot(tlist, pop_11, label='$|11\\rangle$ Population', color='blue', linewidth=2)
ax1.plot(tlist, pop_20, label='$|20\\rangle$ Leakage State', color='red', linestyle='--')
ax1.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('Probability')
ax1.set_title('State Population Dynamics')
ax1.legend()
ax1.grid(True)

# Conditional Phase Accumulation
ax2.plot(tlist, phase_cond_pi, label='Conditional Phase', color='green', linewidth=2)
ax2.axhline(-1.0, color='gray', linestyle=':', label='Target (-1.0 $\pi$)')
ax2.set_xlabel('Time (ns)')
ax2.set_ylabel('Phase Accumulation (Units of $\pi$)')
ax2.set_title('180° ($\pi$) Phase Shift Evolution')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()